# Module 4 — Production Tracing & Experimentation (Arize Phoenix)

Every module so far builds and scores test cases by hand — a `LLMTestCase`, a hand-rolled `trace: List[Dict]`, a `ConversationalTestCase`. That's the right way to *learn* what each metric checks, but it's not how evaluation runs against a live, deployed system. This module is a bridge into that world: **real tracing infrastructure** (every LLM call, tool call, and agent step automatically captured as a span, not appended to a list by hand) and a **formal offline experiment framework** (dataset + task + evaluator as reusable objects, not a one-off script).

Rather than rebuilding this from scratch, this module points at an existing, complete 5-lab course already in this repo — the DeepLearning.AI × Arize *"Evaluating AI Agents"* sequence at `Agent_Evaluation/DeepLearningAI_Arize/` — and explains how each lab maps onto the vocabulary Modules 1–3 already taught. **The labs are left exactly where they are, unmodified**, rather than copied or rewritten into this tutorial folder.

## Why this module is a bridge, not a rebuild

Two reasons this content stays in place instead of being merged in like Modules 1–3 were:

1. **Real folder-scoped dependencies.** Each lab (`Lab 1 - Building your Agent/` through `Lab 5 - Adding Structure to your Evaluations/`) has its own `helper.py`/`utils.py`, local `data/` (a parquet file of toy store transactions) and `images/` used inline in the notebook's own markdown. Copying the cells elsewhere without those files would silently break on the first `%run`-free `from utils import ...`.
2. **A live tracing backend is a real prerequisite, not a mock.** Unlike Module 3's hand-rolled `trace` list, this course's tracing is backed by an actual Phoenix server. As originally authored (on DeepLearning.AI's hosted platform), each lab notebook was pre-wired to a sandbox-provisioned Phoenix endpoint — but `helper.py`'s `get_phoenix_endpoint()` reads `PHOENIX_COLLECTOR_ENDPOINT` from `.env`, so the labs are runnable outside that sandbox too, provided you point that variable at a Phoenix instance you control (see Prerequisites below).

## Prerequisites to run the labs yourself

- A running Phoenix instance — either `phoenix serve` (standalone) or `px.launch_app()` inside a notebook (see [Phoenix's own docs](https://docs.arize.com/phoenix/deployment/environments#notebooks)) — with `PHOENIX_COLLECTOR_ENDPOINT` set in `.env` pointing at it.
- `OPENAI_API_KEY` in `.env` (same as every other module in this tutorial).
- The lab folder's own pinned dependencies (`Agent_Evaluation/DeepLearningAI_Arize/requirements.txt` — notably `arize-phoenix==7.1.1` / `arize-phoenix-otel==0.7.1`, not currently installed in this tutorial's root `.venv`). Given how much version drift this tutorial already hit with `deepeval`'s `Synthesizer` API and `ragas`'s dependency resolution (see `TUTORIAL_PLAN.md`), check those pins against whatever's current before assuming the lab code runs unmodified.

## Concept map: how the 5 labs relate to Modules 1–3

| Lab | What it does | How it maps to earlier modules |
|---|---|---|
| **Lab 1 — Building your Agent** (`L3.ipynb`) | Builds the system under test: a router (OpenAI function calling) over three tools — a database-lookup tool that generates SQL against a local parquet file, a data-analysis tool, and a chart-code-generation tool. No evaluation yet. | Parallel to Module 3's mock/real agent builds (Parts A–C) — this is the "agent under test" half of the same pattern, just a richer 3-tool system instead of the tutorial's toy examples. |
| **Lab 2 — Tracing your Agent** (`L5.ipynb`) | Instruments the Lab 1 agent with real tracing: `OpenAIInstrumentor` auto-captures every LLM call as a span; `agent`/`chain`/`tool`-kind spans are created manually around the router loop and each tool call, all sent to Phoenix. | The production-grade version of Module 3's hand-rolled `trace: List[Dict]` — same underlying idea (a structured record of what happened, step by step), but captured by real instrumentation instead of an `.append()` call in your own code. |
| **Lab 3 — Adding Router & Skill Evaluations** (`L7.ipynb`) | Two new kinds of check: **router evals** (`llm_classify`, an LLM-as-judge, scoring whether the router picked the right tool and extracted the right parameters — queried directly off the spans captured in Lab 2) and **skill evals** (does tool 1's generated SQL look correct? Is tool 2's analysis clear? Does tool 3's generated chart code actually run — a code-based, not LLM-judged, check). | Router evals ≈ Module 2.2's `ToolCorrectnessMetric`/`ArgumentCorrectnessMetric`, done by querying trace spans instead of populating a `ToolCall` object by hand. **Skill evals are new territory** this tutorial hasn't covered elsewhere: judging a *tool's own output quality* (SQL correctness, generated-code runnability, analysis clarity) rather than the RAG/agent's final answer — closest analog is Module 1's generator metrics, but applied per-tool instead of to one final response. |
| **Lab 4 — Adding Trajectory Evaluations** (`L9.ipynb`) | Introduces Phoenix's formal **experiment** abstraction — a `dataset` of examples, a `task` (`run_agent_and_track_path`, which runs the agent and records path length), and an `evaluator` (`evaluate_path_length`, which computes a *convergence score*: did the agent reach the answer efficiently, or wander?). | Conceptually the same question as Module 3's Step-wise Accuracy / Cost-Efficiency proxy (was the path efficient, not just successful) — but running through a reusable `dataset`/`task`/`evaluator` triple that generalizes to *any* agent and *any* metric, instead of a one-off script computed over a single hand-built trace. |
| **Lab 5 — Adding Structure to your Evaluations** (`L11.ipynb`) | Runs multiple evaluators together as one structured experiment (entity correctness, analysis clarity, router correctness), then re-runs the **same experiment against a changed system prompt** to compare versions — plus an optional walkthrough of Phoenix's Playground for interactive prompt iteration. | The "change in prompt, re-run the same eval, compare" pattern is genuinely new relative to Modules 1–3 — none of the earlier notebooks compare two versions of the same system side by side. This is the production answer to "did my prompt change actually help, or did it just feel better?" |

**The throughline:** Labs 1–2 build the system and capture what it does; Labs 3–5 are progressively more structured ways of asking the same questions Modules 1–3 already introduced (tool correctness, trajectory efficiency, outcome quality) — just against a real trace store instead of a hand-built one, and with the added ability to compare *versions* of a system against each other, not just judge one run in isolation.

## Suggested path through the labs

Run in order — each lab builds on artifacts (the instrumented agent, the Phoenix project) created by the one before it:

1. [`Agent_Evaluation/DeepLearningAI_Arize/Lab 1 - Building your Agent/L3.ipynb`](../Agent_Evaluation/DeepLearningAI_Arize/Lab%201%20-%20Building%20your%20Agent/L3.ipynb)
2. [`Agent_Evaluation/DeepLearningAI_Arize/Lab 2 - Tracing your Agent/L5.ipynb`](../Agent_Evaluation/DeepLearningAI_Arize/Lab%202%20-%20Tracing%20your%20Agent/L5.ipynb)
3. [`Agent_Evaluation/DeepLearningAI_Arize/Lab 3 - Adding Router & Skill Evaluations/L7.ipynb`](../Agent_Evaluation/DeepLearningAI_Arize/Lab%203%20-%20Adding%20Router%20%26%20Skill%20Evaluations/L7.ipynb)
4. [`Agent_Evaluation/DeepLearningAI_Arize/Lab 4 - Adding Trajectory Evaluations/L9.ipynb`](../Agent_Evaluation/DeepLearningAI_Arize/Lab%204%20-%20Adding%20Trajectory%20Evaluations/L9.ipynb)
5. [`Agent_Evaluation/DeepLearningAI_Arize/Lab 5 - Adding Structure to your Evaluations/L11.ipynb`](../Agent_Evaluation/DeepLearningAI_Arize/Lab%205%20-%20Adding%20Structure%20to%20your%20Evaluations/L11.ipynb)

A course-level `README.md` sits at `Agent_Evaluation/DeepLearningAI_Arize/README.md`, and an `Appendix – Resources, Tips and Help/` notebook covers platform mechanics (downloading notebooks, accessing helper files) specific to the course's original DeepLearning.AI delivery — skip that one if you're running locally rather than on their platform.

## Summary

- This module is deliberately a **map, not a rebuild** — the 5-lab Arize course already covers this ground thoroughly and depends on real infrastructure (a Phoenix trace store, per-lab local files) that's better left in place than fragmented across a copy.
- **New ideas here that Modules 1–3 didn't cover:** evaluating a tool's own output quality in isolation ("skill evals"), and comparing two versions of a system (a prompt change) against the same experiment rather than judging one run at a time.
- **Ideas you already know, just running on real infrastructure:** tool-selection correctness (Module 2.2), trajectory efficiency (Module 3) — same questions, asked against spans captured by real tracing instead of a hand-built `trace` list.
- Next: [Module 5](14_Capstone_CrewAI_Travel_Planner_Eval.ipynb) — a capstone applying `TaskCompletionMetric` (Module 2.3) to a real, running multi-agent CrewAI application, closing the loop between this tutorial's hand-built examples and a real deployed system.